# Assignment 5: Neural Networks

---

## Task 1) RNN as Language Model

Similar to the n-gram language models in the previous tasks, imagine you have to write another thesis and just want to generate an interesting topic.
In this assignment, you will train and use Recurrent Neural Networks as language models to generate new potential thesis topics.

### Data

Download the `theses.csv` data set from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group.
This dataset consists of approx. 3,000 theses topics chosen by students in the past.
Here are some examples of the file content:

```
27.10.94;14.07.95;1995;intern;Diplom;DE;Monte Carlo-Simulation für ein gekoppeltes Round-Robin-System;
04.11.94;14.03.95;1995;intern;Diplom;DE;Implementierung eines Testüberdeckungsgrad-Analysators für RAS;
01.11.20;01.04.21;2021;intern;Bachelor;DE;Landessprachenerkennung mittels X-Vektoren und Meta-Klassifikation;
```

### Basic Setup

For the assignment on Recurrent Neural Networks, we'll (again) heavily use [PyTorch](https://pytorch.org) as go-to Deep Learning library.
Here, we'll rely on the RNN and Embedding modules already implemented by PyTorch.
You can imagine the Embedding layer as a simple lookup table that stores embeddings of a fixed dictionary and size (quite similar to the Word2Vec parameters we've trained in assignment 2).
Head over to the [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) and [Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) modules to gain some understanding of their functionality.
Code for processing data samples, batching, converting to tensors, etc. can get messy and hard to maintain. 
Therefore, you can use PyTorch's [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html). 
Get familiar with the basics of data handling, as it will help you for upcoming assignments.
As always, you can use [NumPy](https://numpy.org) and [Pandas](https://pandas.pydata.org) for data handling etc.

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [77]:
# Dependencies
import os
import tqdm
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### Prepare the Data

1.1 Spend some time on preparing the dataset. It may be helpful to lower-case the data and to filter for German titles. The format of the CSV-file should be:

```
Anmeldedatum;Abgabedatum;JahrAkademisch;Art;Grad;Sprache;Titel;Abstract
```

1.2 Create the vocabulary from the prepared dataset. You'll need it for the modeling part such as nn.Embedding.

1.3 Create a PyTorch Dataset class which handles your tokenized data with respect to model inputs and labels.

In [78]:
def load_theses_dataset(filepath):
    """Loads all theses instances and returns them as a dataframe."""
    ### YOUR CODE HERE
    
    return pd.read_csv(filepath, sep=";")
    
    ### END YOUR CODE

In [79]:
### Notice: Think about start and end of sentence tokens

def preprocess(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Preprocesses and tokenizes the given theses titles for further use."""
    ### YOUR CODE HERE
    pattern = r"\b[\wäöüÄÖÜß]+(?:[-'][\wäöüÄÖÜß]+)*\b"

    df_german = dataframe[dataframe.Sprache == "DE"].copy()

    tokens = [["<s>"] + re.findall(pattern, str(row).lower(), flags=re.UNICODE) + ["</s>"] for row in df_german.Titel]
    df_german["tokens"] = tokens

    return df_german

    ### END YOUR CODE

In [80]:
df = load_theses_dataset("C:\\Users\\Felix\\PythonProjects\\seqlrn_assignments\\5-nnet_rnn\\data\\theses2022.csv")
df = preprocess(df)

# vocabulary for whole dataframe so there are no embedding errors later
PAD_TOKEN = "<PAD>"
vocabulary = set([PAD_TOKEN])
for title in df.tokens:
    vocabulary.update(title)
vocab_size = len(vocabulary)

word2idx = {
    word: idx for idx, word in enumerate(vocabulary)
}

idx2word = {
    idx: word for word, idx in word2idx.items()
}

In [81]:
### TODO: 1.3 Implement the PyTorch theses dataset
### Notice: It is possible to solve the task without this class.
### Notice: However, with respect to DataLoaders it makes your life easier.

### YOUR CODE HERE

class ThesesDataset(Dataset):
    def __init__(self, data, word2idx):
        self.data, self.labels = [], []

        self.data = data.apply(lambda title: [word2idx[token] for token in title[:-1]]).to_list()
        self.labels = data.apply(lambda title: [word2idx[token] for token in title[1:]]).to_list()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.data[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )
    
### END YOUR CODE

### Train and Evaluate

2.1 Implement the RNN Language Model. Therefore, you can use the nn.Module and overwrite the forward function. For the embedding layer you can either use the embeddings learned from the previous word2vec assignment or train the `nn.Embedding` module and corresponding parameters from scratch.

2.2 Implement the functionality to train your model with the train dataset.

2.3 Implement the functionality to evaluate your model with the test dataset.

2.4 Perform a train-test-split for your theses data, train the RNN Language Model and evaluate the loss & perplexity.

In [82]:
### TODO: 2.1 Implement the RNN Language Model (nn.Module)

### YOUR CODE HERE

class RNN_LM(nn.Module):
    def __init__(self, word2idx, embedding_dim, hidden_dim, num_layers):
        super(RNN_LM, self).__init__()

        self.embedding = nn.Embedding(len(word2idx), embedding_dim, padding_idx=word2idx["<PAD>"])
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )      

        self.fc = nn.Linear(hidden_dim, len(word2idx))

    def forward(self, X, hidden=None):
        embeddings = self.embedding(X) 
        outputs, hidden = self.rnn(embeddings, hidden)
        logits = self.fc(outputs)
        return logits, hidden

### END YOUR CODE

In [83]:
### TODO: 2.2 Implement the train functionality
### Notice: If you want, you can also combine train and eval functionality

def train(model: RNN_LM, dataloader: DataLoader, optimizer: optim.Optimizer, criterion, device):
    """Trains the RNN-LM for one epoch."""
    ### YOUR CODE HERE
    epoch_loss = 0
    model.to(device)
    model.train()

    for sample, targets in dataloader:
        optimizer.zero_grad()

        sample = sample.to(device)
        targets = targets.to(device)
        preds, _ = model(sample)
        loss = criterion(preds.squeeze(0), targets.squeeze(0))
        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss
    ### END YOUR CODE

In [84]:
### TODO: 2.3 Implement the evaluation functionality
### Notice: If you want, you can also combine train and eval

def eval(model: RNN_LM, dataloader: DataLoader, criterion, device):
    """Evaluates the optimized RNN-LM."""
    ### YOUR CODE HERE
    eval_loss = 0
    all_preds = []
    model.to(device)
    model.eval()

    with torch.no_grad():
        for sample, targets in dataloader:
            sample = sample.to(device)
            targets = targets.to(device)
            
            preds, _ = model(sample)
            loss = criterion(preds.squeeze(0), targets.squeeze(0))

            eval_loss += loss.item()
            
            all_preds.append(torch.argmax(torch.softmax(preds, dim=1), dim=0).tolist())

    return eval_loss, all_preds
    ### END YOUR CODE

In [85]:
### TODO: 2.4 Initialize and train the RNN Language Model for X epochs

# For split reproducibility
# Use 5-fold cross validation
SEED = 666

EPOCHS = 25

print(torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # 'cpu', 'mps' or 'cuda'

LABEL_COL = "Grad"

LEARN_RATE = 0.0001

### YOUR CODE HERE
data = df[df["tokens"].apply(lambda x: len(x) > 1)].reset_index(drop=True)["tokens"]


kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
kf.get_n_splits(data)

# Your language model
rnn = RNN_LM(word2idx, embedding_dim=300, hidden_dim=64, num_layers=1)

# Your loss function
criterion = nn.CrossEntropyLoss()

# Your optimizer (optim.SGD should be okay)
rnn_optimizer = optim.Adam(rnn.parameters(), lr=LEARN_RATE)

for i, (train_index, test_index) in enumerate(kf.split(data)):
    print(f"Fold {i}:")

    # Use batch_size=1 if you want to avoid padding handling
    train_dataset = ThesesDataset(data.iloc[train_index], word2idx)
    train_dataloader = DataLoader(train_dataset)

    # Use batch_size=1 if you want to avoid padding handling
    test_dataset = ThesesDataset(data.iloc[test_index], word2idx)
    test_dataloader = DataLoader(test_dataset)

    for e in range(1, EPOCHS+1):
        train_loss_rnn = train(rnn, train_dataloader, rnn_optimizer, criterion, DEVICE)
        print(f"    Epoch {e} RNN loss: {train_loss_rnn}")

    rnn_eval_loss, rnn_preds = eval(rnn, test_dataloader, criterion, DEVICE)
    print(f"    Fold {i} RNN eval loss: {rnn_eval_loss}")

### END YOUR CODE

False
Fold 0:
    Epoch 1 RNN loss: 19002.318497657776
    Epoch 2 RNN loss: 14564.29495882988
    Epoch 3 RNN loss: 13862.344746112823
    Epoch 4 RNN loss: 13431.978437662125
    Epoch 5 RNN loss: 13086.809423446655
    Epoch 6 RNN loss: 12788.047686338425
    Epoch 7 RNN loss: 12523.153978347778
    Epoch 8 RNN loss: 12282.054089546204
    Epoch 9 RNN loss: 12058.64121055603
    Epoch 10 RNN loss: 11849.272426366806
    Epoch 11 RNN loss: 11650.279246807098
    Epoch 12 RNN loss: 11460.259185552597
    Epoch 13 RNN loss: 11278.400779008865
    Epoch 14 RNN loss: 11104.069200754166
    Epoch 15 RNN loss: 10936.772469520569
    Epoch 16 RNN loss: 10775.694633483887
    Epoch 17 RNN loss: 10620.033415079117
    Epoch 18 RNN loss: 10469.410229206085
    Epoch 19 RNN loss: 10323.475093603134
    Epoch 20 RNN loss: 10181.765121102333
    Epoch 21 RNN loss: 10043.851439714432
    Epoch 22 RNN loss: 9909.350878834724
    Epoch 23 RNN loss: 9777.978073120117
    Epoch 24 RNN loss: 9649.56108

### Generate Titles

3.1 Use the trained RNN Language Model to generate theses titles. How can you sample the next tokens?

3.2 Compare your results with n-gram language models (e.g., n=4). Of course, you can use a library such as NLTK toolkit
- What perplexity does a regular 4-gram have on the same split? 
- Compare the generated titles from the 4-gram and RNN-LM. Do you think the n-gram titles are better?

In [88]:
### TODO: 3.1 Generate titles with the trained RNN Language Model

def generate(model, word2idx, idx2word, title_length=15):
    ### YOUR CODE HERE
    idx_sequence = [word2idx[word] for word in ["<s>"]]

    model.eval()
    hidden = None

    while len(idx_sequence) <= title_length and idx2word[idx_sequence[-1]] != "</s>":
        output, hidden = model(torch.tensor(idx_sequence[-1], dtype=torch.long).unsqueeze(0), hidden)
        new_token = torch.argmax(torch.softmax(output[-1], dim=0), dim=0).tolist()
        idx_sequence.append(new_token)

    return [idx2word[idx] for idx in idx_sequence]
    ### END YOUR CODE

for i in range(10):
    generated_title = generate(rnn, word2idx, idx2word)
    print(" ".join(generated_title))

<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische implementierung eines systems zur automatisierten erstellung von teilnehmerlisten </s>
<s> konzeption und prototypische impleme

In [123]:
### TODO: 3.2 Generate titles with the trained n-gram language model

### YOUR CODE HERE
import random
from collections import Counter
import nltk

def build_n_grams(data, n=4):
    n_grams_probs = []

    for i in range(1, n+1):
        n_grams = []
        
        for tokenized_title in data:
            n_grams.extend(list(nltk.ngrams(tokenized_title, n=i)))
        n_grams_counter = Counter(n_grams)
        n_grams_count = len(n_grams_counter)
        n_grams_probs.append({gram: count / n_grams_count for gram, count in n_grams_counter.items()})

    return n_grams_probs


def generate_ngrams(probs, title_length=15, n=4):
    generated_sequence = ["<s>"]
    i = len(generated_sequence) + 1

    while len(generated_sequence) < title_length and generated_sequence[-1] != "</s>":
        current_n = min(i, n) # go up until n is reached
        context = tuple(generated_sequence[-(current_n - 1):])
        print(probs[current_n - 1].keys())
        print(context)
        candidates = {gram[-1]: prob for gram, prob in probs[current_n - 1].items() if gram[:current_n-1] == context}
        print(candidates)
        new_token = random.choices(list(candidates.keys()))
        generated_sequence.append(new_token[0])
        i += 1 

    return generated_sequence


n_grams_probs = build_n_grams(data, n=4)
for i in range(10):
    generated_title = generate_ngrams(n_grams_probs)
    print(" ".join(generated_title))

### END YOUR CODE

dict_keys([('<s>', 'email'), ('email', 'am'), ('am', 'beispiel'), ('beispiel', 'smtp'), ('smtp', 'im'), ('im', 'internet'), ('internet', '</s>'), ('<s>', 'einführung'), ('einführung', 'des'), ('des', 'configuration'), ('configuration', 'management-systems'), ('management-systems', 'pcms'), ('pcms', 'zur'), ('zur', 'strukturierten'), ('strukturierten', 'versions-release'), ('versions-release', 'und'), ('und', 'änderungskontrolle'), ('änderungskontrolle', 'in'), ('in', 'projekten'), ('projekten', 'der'), ('der', 'abteilung'), ('abteilung', 'information-systems'), ('information-systems', 'der'), ('der', 'firma'), ('firma', 'motorola'), ('motorola', '</s>'), ('<s>', 'analyse'), ('analyse', 'und'), ('und', 'leistungsvergleich'), ('leistungsvergleich', 'von'), ('von', 'zwei'), ('zwei', 'echtzeitsystemen'), ('echtzeitsystemen', 'für'), ('für', 'eingebettete'), ('eingebettete', 'anwendungen'), ('anwendungen', '</s>'), ('<s>', 'erfassung'), ('erfassung', 'und'), ('und', 'automatische'), ('autom

IndexError: list index out of range